# The spelled-out intro to language modeling: building makemore

Based on [Andrej Karpathy's video lecture](https://www.youtube.com/watch?v=PaCmpygFfXo) from the "Neural Networks: Zero to Hero" series.

---

## intro

Like micrograd before it, makemore is a repository on GitHub. Just like with micrograd, we're going to build it out step by step and spell everything out. We're going to build it out slowly and together.

**What is makemore?** As the name suggests, makemore makes more of things that you give it. `names.txt` is an example dataset - a very large dataset of names with about 32,000 names found randomly on a government website.

If you train makemore on this dataset, it will learn to make more things that sound name-like but are actually unique names. Maybe if you have a baby and you're looking for a cool new sounding unique name, makemore might help you. Here are some example generations from the neural network once we train it: *Dontel*, *Irot*, *Zhendi* - they sound name-like but they're not actual names.

**Under the hood, makemore is a character-level language model.** This means it treats every single line as an example, and within each example it treats them as sequences of individual characters. For example, "reese" is the sequence r, e, e, s, e.

Being a character-level language model means it's modeling those sequences of characters and it knows how to predict the next character in a sequence.

We're going to implement a large number of character-level language models in terms of the neural networks involved:
- Very simple **bigram** and bag-of-words models
- **Multilayer perceptrons (MLPs)**
- **Recurrent neural networks (RNNs)**
- Modern **Transformers**

In fact, the transformer that we will build will be basically equivalent to GPT-2. That's kind of a big deal - it's a modern network, and by the end of the series you will understand how that works on the level of characters.

For now, we have to start here: **character-level language modeling**. Let's go.

## reading and exploring the dataset

We are starting with a completely blank Jupyter notebook page. The first thing is I would like to basically load up the dataset `names.txt`. We're going to open up `names.txt` for reading and we're going to read in everything into a massive string. And then because it's a massive string we'd only like the individual words and put them in a list, so let's call `splitlines()` on that string to get all of our words as a Python list of strings.

In [ ]:
words = open('names.txt', 'r').read().splitlines()

In [ ]:
words[:10]

So basically we can look at for example the first 10 words and we have that it's a list of emma, olivia, ava and so on. This list actually makes me feel that this is probably sorted by frequency, but okay.

These are the words. Now we'd like to actually learn a little bit more about this dataset. Let's look at the total number of words - we expect this to be roughly 32,000.

In [ ]:
len(words)

In [ ]:
min(len(w) for w in words)  # shortest word

In [ ]:
max(len(w) for w in words)  # longest word

So the shortest word will be length 2 and the longest word will be 15 characters.

## exploring the bigrams in the dataset

Now let's think through our very first language model. As I mentioned, a character-level language model is predicting the next character in a sequence given already some concrete sequence of characters before it.

**Here's an important realization:** every single word here like "isabella" is actually quite a few examples packed into that single word. What is the existence of a word like "isabella" in the dataset telling us?

- The character 'i' is a very likely character to come first in the sequence of a name
- The character 's' is likely to come after 'i'
- The character 'a' is likely to come after "is"
- The character 'b' is very likely to come after "isa"
- And so on, all the way to 'a' following "isabel"

And there's **one more example** packed in here: after "isabella", the word is very likely to **end**. That's one more explicit piece of information that we have to be careful with.

So there's a lot packed into a single individual word in terms of the statistical structure of what's likely to follow in these character sequences. And of course we don't have just an individual word - we actually have 32,000 of them! So there's a lot of structure here to model.

### The bigram language model

In the **bigram language model**, we're always working with just two characters at a time. We're only looking at one character that we are given and we're trying to predict the next character in the sequence.

What characters are likely to follow 'a'? What characters are likely to follow 'r'? We're modeling that kind of little local structure and we're forgetting that we may have a lot more information. We're always just looking at the previous character to predict the next one. It's a very simple and weak language model, but it's a great place to start.

### The zip() trick

A cute way to iterate over consecutive characters in Python:

In [ ]:
for w in words[:1]:  # just the first word for now
    chs = ['<S>'] + list(w) + ['<E>']
    for ch1, ch2 in zip(chs, chs[1:]):
        print(ch1, ch2)

The reason this works: `w` is the string "emma", `w[1:]` is the string "mma". `zip` takes two iterators and pairs them up, creating an iterator over the tuples of their consecutive entries. If one of these lists is shorter than the other, it will just halt and return.

So basically that's why we return (e,m), (m,m), (m,a). But then because the second iterator runs out of elements, zip just ends - and that's why we only get these tuples. Pretty cute.

We need special start `<S>` and end `<E>` tokens because we know that 'e' is very likely to come first, and we know that 'a' in this case is coming last.

## counting bigrams in a python dictionary

In order to learn the statistics about which characters are likely to follow other characters, the simplest way in the bigram language model is to simply do it by **counting**. We're basically just going to count how often any one of these combinations occurs in the training set in these words.

We're going to need some kind of a dictionary that's going to maintain some counts for every one of these bigrams:

In [ ]:
b = {}
for w in words:
    chs = ['<S>'] + list(w) + ['<E>']
    for ch1, ch2 in zip(chs, chs[1:]):
        bigram = (ch1, ch2)
        b[bigram] = b.get(bigram, 0) + 1

`b.get(bigram, 0)` is basically the same as `b[bigram]` but in the case that bigram is not in the dictionary `b`, we would like to by default return 0, plus 1. This will basically add up all the bigrams and count how often they occur.

Let's look at the most common and least common bigrams. This kind of grows in Python, but the way to do this - the simplest way I like - is we just use `b.items()`. This returns tuples of (key, value), in this case the keys are the character bigrams and the values are the counts. We want to sort by the values (the counts), so we use `key=lambda kv: -kv[1]`:

In [ ]:
sorted(b.items(), key=lambda kv: -kv[1])[:20]

We see that 'n' was very often an ending character (many many times), and apparently 'n' almost always follows 'a' - that's a very likely combination.

## counting bigrams in a 2D torch tensor ("training the model")

It's actually going to be significantly more convenient for us to keep this information in a **2D array** instead of a Python dictionary. We're going to store this information in a 2D array where:
- Rows are the first character of the bigram
- Columns are the second character
- Each entry tells us how often that first character is followed by the second character

The array representation we're going to use is PyTorch. PyTorch is a deep learning neural network framework, but part of it is also `torch.tensor` which allows us to create multi-dimensional arrays and manipulate them very efficiently.

In [ ]:
import torch

We have 26 letters of the alphabet and then we have a special token. We're going to use just a single `.` token instead of `<S>` and `<E>` - it's cleaner. So we want a 27x27 array.

First we need a lookup table from characters to integers, and vice versa:

In [ ]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}  # a=1, b=2, ..., z=26
stoi['.'] = 0  # special token at position 0
itos = {i:s for s,i in stoi.items()}
print(stoi)
print(itos)

I like having the special `.` token at position 0 and I'd like to offset all the other letters by 1 - I find that a little bit more pleasing.

Now let's create the counts matrix:

In [ ]:
N = torch.zeros((27, 27), dtype=torch.int32)

for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        N[ix1, ix2] += 1

We're using `dtype=torch.int32` because we are going to represent counts (integers, not floats).

## visualizing the bigram tensor

Let's visualize this array. I wrote a bunch of code to make this look nice:

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

plt.figure(figsize=(16,16))
plt.imshow(N, cmap='Blues')
for i in range(27):
    for j in range(27):
        chstr = itos[i] + itos[j]
        plt.text(j, i, chstr, ha="center", va="bottom", color='gray')
        plt.text(j, i, N[i, j].item(), ha="center", va="top", color='gray')
plt.axis('off');

Every cell shows the bigram characters and how many times that bigram occurs in the dataset.

- The first row (`.` row) shows counts for first letters - how often each letter starts a word
- The first column (`.` column) shows counts for last letters - how often each letter ends a word  
- The interior shows which characters follow which in the middle of words

Note: when you index into these arrays, you get a tensor back. To get the actual integer, use `.item()`.

## sampling from the model

This array actually has all the information necessary for us to sample from this bigram character-level language model. Roughly speaking, we're going to follow these probabilities and counts and start sampling from the model.

In the beginning, we start with the `.` (start token). To sample the first character of a name, we're looking at the first row - the counts for what characters are likely to start a word.

First, we need to convert counts to probabilities:

In [ ]:
# Get the first row
p = N[0].float()
p = p / p.sum()
p

We convert to float and divide by the sum. Now we have a nice proper probability distribution - it sums to 1 and gives us the probability for any single character to be the first character of a word.

### torch.multinomial

To sample from these distributions, we use `torch.multinomial`. It takes probabilities and returns integers sampled according to that probability distribution. We use a Generator object to make everything deterministic - you'll get the exact same results that I'm getting.

In [ ]:
g = torch.Generator().manual_seed(2147483647)
p = torch.rand(3, generator=g)
p = p / p.sum()
print(p)
print(torch.multinomial(p, num_samples=20, replacement=True, generator=g))

You'll notice the probability for the first element is ~60%, so in 20 samples we'd expect 60% to be zeros. We'd expect ~30% ones and very few twos (~10%).

Now let's put it all together to sample names:

In [ ]:
# First, let's create the probability matrix P
P = (N+1).float()  # +1 for smoothing (explained later)
P /= P.sum(1, keepdims=True)  # normalize each row

In [ ]:
g = torch.Generator().manual_seed(2147483647)

for i in range(5):
    out = []
    ix = 0  # start with '.'
    while True:
        p = P[ix]
        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        out.append(itos[ix])
        if ix == 0:  # end token
            break
    print(''.join(out))

I'll be honest with you - this doesn't look right. I spent a few minutes convincing myself that it actually IS right.

The reason these samples are so terrible is that **the bigram language model is actually just really terrible**. When it generates 'h' as a complete name, the model doesn't know that 'h' is the very first character. All it knows is that 'h' was the previous character. How likely is 'h' to be the last character? Well, it's somewhat likely, so it ends the word.

It doesn't know that there were other characters before it or not. That's why it generates all these nonsense names.

To convince yourself this is working: if we used a uniform distribution (everything equally likely), we'd get complete garbage. The bigram model is at least *more* name-like - it's doing something. It's just that bigrams alone aren't enough context.

## efficiency! vectorized normalization of the rows, tensor broadcasting

What we're doing in the loop above is fetching a row of N from the counts matrix, converting to float, dividing - doing this every single iteration. That's extremely inefficient and wasteful.

I'd like to prepare a matrix `P` that will have the probabilities in it upfront. This is not just for efficiency - I'd like us to practice these n-dimensional tensors and their manipulation, especially something called **broadcasting**.

We're going to have to become very good at these tensor manipulations because if we're going to build all the way to Transformers, we're going to be doing some pretty complicated array operations.

### The keepdims bug (important!)

I want to **scare you a little bit** here. I strongly encourage you to read through broadcasting semantics and treat this with respect. It's not something to play fast and loose with.

In [ ]:
# The RIGHT way:
P = N.float()
P /= P.sum(1, keepdims=True)

# Check: each row should sum to 1
print("First row sum:", P[0].sum().item())

In [ ]:
# The WRONG way (without keepdims):
P_wrong = N.float()
P_wrong /= P_wrong.sum(1)  # BUG!

# This normalizes COLUMNS instead of ROWS!
print("First row sum (wrong):", P_wrong[0].sum().item())
print("First column sum (wrong):", P_wrong[:, 0].sum().item())

What happened? 

- With `keepdims=True`: `P.sum(1, keepdims=True)` has shape (27, 1) - a **column vector**. Broadcasting copies it horizontally, correctly normalizing each row.

- Without `keepdims`: `P.sum(1)` has shape (27,). In broadcasting, this becomes (1, 27) - a **row vector**. This gets copied vertically, normalizing columns instead of rows!

This is a very subtle bug. The code runs without error but gives completely wrong results. The fix is just one parameter, but missing it means your model is broken in a way that's very hard to debug.

**Pretty subtle - hopefully this helps to scare you that you should have respect for broadcasting, be careful, check your work, and understand how it works under the hood.**

## loss function (the negative log likelihood of the data under our model)

Now I'd like to somehow evaluate the quality of this model - summarize it into a **single number**. How good is it at predicting the training set?

We have 27 possible characters. If everything was equally likely, you'd expect all probabilities to be about 4% (1/27). Anything above 4% means we've learned something useful from the bigram statistics.

### Likelihood and Log Likelihood

The **likelihood** is the product of all the probabilities the model assigns to the correct next characters. But products of many small numbers become tiny and unwieldy.

So we use **log likelihood** instead:
- `log(a × b × c) = log(a) + log(b) + log(c)`
- Products become sums - much more manageable!

The log function is monotonic: log(1) = 0, and as probability decreases, log becomes more negative (toward -∞ at probability 0).

### Negative Log Likelihood (NLL)

We want a **loss function** where **lower is better** (we're trying to minimize). Since log likelihood is negative, we just flip the sign:

- **Negative log likelihood** = our loss
- Lower NLL = better model
- Minimize NLL = maximize likelihood

In [ ]:
# Recreate P with smoothing
P = (N+1).float()
P /= P.sum(1, keepdims=True)

log_likelihood = 0.0
n = 0

for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        prob = P[ix1, ix2]
        logprob = torch.log(prob)
        log_likelihood += logprob
        n += 1

print(f'{log_likelihood=}')
nll = -log_likelihood
print(f'{nll=}')
print(f'Average NLL (loss): {nll/n}')

Our loss is about 2.45. The lower it is, the better. The job of training is to find parameters that minimize this negative log likelihood loss - that would be a high quality model.

## model smoothing with fake counts

What if we encounter a bigram like "jq" that never appeared in training? The probability would be 0, and log(0) = -infinity. That's undesirable.

**Model smoothing** is a simple fix: we add fake counts (like +1) to everything. This ensures no probability is exactly zero.

- Add more = more uniform distribution
- Add less = more peaked distribution
- Adding +1 is a decent default

In [ ]:
# P = (N+1).float()  # This is what we already have!
# The +1 ensures no zeros

---

# PART 2: the neural network approach

---

## PART 2: the neural network approach: intro

We've arrived at the bigram model explicitly by doing something that felt sensible - counting and normalizing. Now I'd like to take an **alternative approach**. We will end up in a very similar position, but the approach will look very different.

I would like to cast the problem of bigram character-level language modeling into the **neural network framework**.

Our neural network will:
- Receive a single character as input
- Have some weights/parameters W
- Output the probability distribution over the next character
- Be trained using gradient-based optimization to minimize the loss

The beautiful thing is that we can use the **exact same loss function** (negative log likelihood) to guide the optimization.

## creating the bigram dataset for the neural net

First, let's create the training set - all the bigrams as (input, target) pairs:

In [ ]:
# Create the training set
xs, ys = [], []

for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        xs.append(ix1)
        ys.append(ix2)

xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.nelement()
print(f'Number of examples: {num}')

The single word "emma" has 5 bigrams: (.e), (em), (mm), (ma), (a.). So emma alone gives us 5 training examples. With 32,000 words, we get about 228,000 examples total.

## feeding integers into neural nets? one-hot encodings

You can't just plug an integer like 13 into a neural network. Neural nets are made up of neurons with weights that act **multiplicatively** on inputs: `wx + b`. It doesn't make sense for an input neuron to take on integer values.

Instead, we use **one-hot encoding**: take an integer like 13 and create a vector that is all zeros except for the 13th position, which is 1.

```
13 → [0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0]
```

In [ ]:
import torch.nn.functional as F

xenc = F.one_hot(xs, num_classes=27).float()
print(f'Shape: {xenc.shape}')

In [ ]:
plt.imshow(xenc[:5])

We've encoded all examples into vectors. Each row is now an example that can feed into a neural net - the appropriate bit is turned on as a 1 and everything else is 0.

**Note on dtype:** `F.one_hot` returns integers by default. We cast to float because that's what neural nets need.

## the "neural net": one linear layer of neurons implemented with matrix multiplication

Now let's construct our first neuron. These neurons perform `wx + b` - a dot product. We can do this efficiently with matrix multiplication.

We're going to have 27 neurons (one for each possible next character), each receiving 27 inputs (the one-hot encoded previous character).

In [ ]:
# Initialize random weights
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g, requires_grad=True)

In [ ]:
# Forward pass through the linear layer
logits = xenc @ W  # (num, 27) @ (27, 27) -> (num, 27)
print(f'Logits shape: {logits.shape}')

Using matrix multiplication, we efficiently evaluate the dot product between all input examples in a batch and all 27 neurons. The neurons have their weights stored in the columns of W.

**What does `xenc @ W` actually compute?** For a one-hot input with a 1 at position 5, multiplying by W essentially **plucks out the 5th row of W**. The other zeros contribute nothing. This is exactly like indexing into a table!

## transforming neural net outputs into probabilities: the softmax

The neural net outputs can be positive or negative - they're not probabilities yet. We interpret them as **log-counts** (logits) and:

1. **Exponentiate** to get positive "counts"
2. **Normalize** to get probabilities that sum to 1

This is called **softmax** - a very commonly used layer in neural nets.

In [ ]:
# Softmax
counts = logits.exp()  # exponentiate to get positive numbers
probs = counts / counts.sum(1, keepdims=True)  # normalize rows

print(f'Probs shape: {probs.shape}')
print(f'First row sum: {probs[0].sum().item()}')

For every example, we now have a probability distribution over the next character. All of these operations are differentiable - we can backpropagate through them!

## vectorized loss

Now we compute the loss - the average negative log likelihood. We need to pluck out the probability the model assigns to the **correct** next character for each example:

In [ ]:
# Get probability of correct character for each example
loss = -probs[torch.arange(num), ys].log().mean()
print(f'Loss: {loss.item()}')

`probs[torch.arange(num), ys]` uses fancy indexing: for each row `i`, it gets the element at column `ys[i]` - the probability of the correct next character. Then we take log, negate, and average.

## backward and update, in PyTorch

Now for the magic. PyTorch keeps track of all operations and builds a computational graph (just like we did manually in micrograd). We can call `loss.backward()` to compute gradients.

In [ ]:
# Backward pass
W.grad = None  # reset gradients
loss.backward()

print(f'W.grad shape: {W.grad.shape}')

`W.grad` now contains the gradient of the loss with respect to every element of W. Positive gradient = increasing that weight increases the loss. Negative = decreasing the loss.

In [ ]:
# Update weights
W.data += -50 * W.grad

We nudge weights in the opposite direction of the gradient (to minimize loss). The learning rate (50) controls step size.

## putting everything together

Let's put the full training loop together:

In [ ]:
# Initialize weights
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g, requires_grad=True)

# Training loop
for k in range(100):
    # Forward pass
    xenc = F.one_hot(xs, num_classes=27).float()
    logits = xenc @ W
    counts = logits.exp()
    probs = counts / counts.sum(1, keepdims=True)
    loss = -probs[torch.arange(num), ys].log().mean() + 0.01*(W**2).mean()
    
    
    # Backward pass
    W.grad = None
    loss.backward()
    
    # Update
    W.data += -50 * W.grad
    
    if k % 10 == 0:
        print(f'Iteration {k}: loss = {loss.item():.4f}')

The loss decreases from ~3.8 (random, like guessing from 27 characters: log(27) ≈ 3.3) down to ~2.45 - exactly what we got from counting! The neural network learned the same bigram statistics via gradient descent.

## note 1: one-hot encoding really just selects a row of the next Linear layer's weight matrix

Here's an important insight. When we do `xenc @ W` with a one-hot vector that has a 1 at position 5:

Because of how matrix multiplication works, **we're just plucking out the 5th row of W**. The other zeros contribute nothing.

This is **exactly** what happened in the counting approach! We took the first character and indexed into a row of our probability matrix. The neural net is doing the same thing - the weights W are essentially the **log counts**.

After training:
- `W` ≈ log of the count matrix
- `exp(W)` ≈ the count matrix itself

The difference is HOW we arrived there:
- **Counting**: Populated counts by explicitly counting bigrams
- **Neural net**: Initialized randomly, let the loss guide us to the same answer

That's why both approaches give the same loss and samples!

## note 2: model smoothing as regularization loss

Remember **model smoothing** from the counting approach - adding fake counts to make the distribution more uniform?

The neural network has an equivalent: **L2 regularization** (also called weight decay).

Here's the insight:
- If all entries of W are **zero**: logits are all zero → exp(0) = 1 → all counts = 1 → **uniform probabilities**
- Making W closer to zero → more uniform distribution

We can **incentivize** W to be near zero by adding a penalty term:

```python
regularization_loss = 0.01 * (W**2).mean()
total_loss = nll_loss + regularization_loss
```

Squaring removes signs. If W is zero, regularization loss is zero. If W has large values, you accumulate loss.

- **Stronger regularization** → more uniform predictions (like adding more fake counts)
- **Weaker regularization** → more peaked predictions (trusting the data more)

This is kind of cool - the gradient-based framework has a natural equivalent to smoothing!

## sampling from the neural net

In [ ]:
# Sample from the trained neural network
g = torch.Generator().manual_seed(2147483647)

for i in range(5):
    out = []
    ix = 0
    while True:
        xenc = F.one_hot(torch.tensor([ix]), num_classes=27).float()
        logits = xenc @ W
        counts = logits.exp()
        p = counts / counts.sum(1, keepdims=True)
        
        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        out.append(itos[ix])
        if ix == 0:
            break
    print(''.join(out))

Kind of anticlimactic... or climactic, depending how you look at it! We get the **exact same results** as the counting-based model. 

This isn't a bug - it's the point. These are **identical models**. Not only do they achieve the same loss, but `W` literally becomes the log counts. We just arrived at the same answer in a very different way.

## conclusion

We've actually covered a lot of ground. We introduced the **bigram character-level language model**. We saw:

- How to **train** the model (count bigrams and normalize)
- How to **sample** from the model (iteratively sample next character)
- How to **evaluate** the model (negative log likelihood loss)

Then we trained the model in **two completely different ways** that actually get the same result:

| Counting Approach | Neural Network Approach |
|-------------------|------------------------|
| Count bigrams explicitly | Learn via gradient descent |
| Normalize rows → probabilities | Softmax → probabilities |
| Smoothing = add fake counts | Regularization = push W toward 0 |
| Direct, interpretable | Flexible, scalable |

### Why the neural network approach matters

For bigrams, the counting approach is simpler. So why bother with neural networks?

Right now our neural network is super simple: single previous character → single linear layer → logits. But this is about to **complexify**.

In follow-up videos, we're going to:
- Take **more and more** previous characters as context
- Feed them into increasingly sophisticated neural nets
- But the output is always the same: logits that get normalized the same way
- The loss function stays identical
- The gradient-based framework stays identical

It's just that the neural net will complexify all the way to **Transformers**.

### The fundamental limitation of counting

The counting approach works for bigrams because there are only 27×27 = 729 possible pairs - we can store them all in a table.

But what about:
- **Trigrams?** 27³ = 19,683 combinations
- **4-grams?** 27⁴ = 531,441 combinations
- **10-character context?** 27¹⁰ = way too many!

We can't keep everything in a table anymore. This is fundamentally an **unscalable approach**.

The neural network approach is significantly more scalable - it can **generalize** without storing every combination explicitly. That's where we'll be digging next.

So that's going to be pretty awesome. I'm looking forward to it!